In [ ]:
import requests
import numpy as np
import os
from pathlib import Path

# Only download 26 July 2026
target_day = "20260726"

year_url = (
    "https://data-portal.s5p-pal.com/api/s5p-l3/aai/day/2026"
)

catalog = requests.get(year_url).json().get('links')
catalog_url= [link for link in catalog if link["rel"] == "item" and f"day-{target_day}" in link["href"]]
catalog_array = np.array([link['href'] for link in catalog_url])

if len(catalog_array) == 0:
    raise ValueError(f"No S5P product found for {target_day}")

os.makedirs("Output/s5p", exist_ok=True)

for item_url in catalog_array:

    item = requests.get(item_url).json()

    # Select the NetCDF asset
    download_url = item["assets"].get('product').get('href')
    output_path = Path(download_url.split("/")[-1])
    local_name = item["assets"].get('product').get('file:local_path')
    local_name= Path("Output/s5p/" + local_name)

    # Skip products that were already downloaded
    if os.path.exists(local_name):
        print(f"Already exists: {local_name}")
        continue

    with requests.get(download_url, stream=True) as response:
        response.raise_for_status()
        with output_path.open("wb") as file:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    file.write(chunk)

    print(f"Downloaded: {output_path}")
    print(f"Size: {output_path.stat().st_size / 1024**2:.2f} MB")

    os.rename(output_path,local_name)

In [ ]:
import re
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# -----------------------------
# Config
# -----------------------------
data_dir = Path("Output/s5p")
figure_folder = Path("Output/figures")
figure_folder.mkdir(parents=True, exist_ok=True)
target_day = pd.Timestamp("2026-07-26")

bordeaux_lon, bordeaux_lat = -0.5792, 44.8378
left = bordeaux_lon - 1.5
right = bordeaux_lon + 2.5
bottom = bordeaux_lat - 1.5
top = bordeaux_lat + 1.5

var_name = "absorbing_aerosol_index"
lat_name = "latitude"
lon_name = "longitude"

# -----------------------------
# Find the file for the target day
# -----------------------------
def extract_date_from_name(path: Path):
    m = re.search(r"(20\d{6})", path.name)  # YYYYMMDD
    if not m:
        return None
    try:
        return pd.to_datetime(m.group(1), format="%Y%m%d")
    except Exception:
        return None

day_files = [p for p in sorted(data_dir.glob("*.nc")) if extract_date_from_name(p) == target_day]
if not day_files:
    raise FileNotFoundError(f"No S5P file for {target_day.date()} in {data_dir}")
day_path = day_files[0]
print(f"Using: {day_path}")

# -----------------------------
# Read the Bordeaux region
# -----------------------------
with xr.open_dataset(day_path) as ds:
    da = ds[var_name].squeeze(drop=True)

    # Keep first index for extra dims
    for dim in list(da.dims):
        if dim not in (lat_name, lon_name):
            da = da.isel({dim: 0})

    # Ensure ascending coords for slicing
    if da[lat_name].values[0] > da[lat_name].values[-1]:
        da = da.sortby(lat_name)
    if da[lon_name].values[0] > da[lon_name].values[-1]:
        da = da.sortby(lon_name)

    da_region = da.sel(
        **{
            lon_name: slice(left, right),
            lat_name: slice(bottom, top),
        }
    ).transpose(lat_name, lon_name).load()

if da_region.size == 0:
    raise ValueError("No S5P data in the selected map area")

lon = da_region[lon_name].values
lat = da_region[lat_name].values
z = da_region.values

finite_z = z[np.isfinite(z)]
vmin = np.nanpercentile(finite_z, 5) if finite_z.size else None
vmax = np.nanpercentile(finite_z, 95) if finite_z.size else None

# -----------------------------
# Figure (same layout as the GEDI and S3 maps)
# -----------------------------
map_projection = ccrs.PlateCarree()
figure, map_ax = plt.subplots(
    figsize=(12, 10),
    subplot_kw={"projection": map_projection},
    constrained_layout=True,
)
map_ax.add_feature(cfeature.OCEAN, facecolor="#dbeaf2", zorder=0)
map_ax.add_feature(cfeature.LAND, facecolor="#eeeeea", zorder=0)
map_ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor="#444444", zorder=3)
map_ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor="#666666", zorder=3)
map_ax.set_extent([left, right, bottom, top], crs=map_projection)

raster = map_ax.pcolormesh(
    lon,
    lat,
    z,
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
    shading="auto",
    alpha=0.65,
    transform=map_projection,
    zorder=2,
)

map_ax.scatter(
    [bordeaux_lon],
    [bordeaux_lat],
    c="red",
    s=50,
    edgecolors="white",
    linewidths=1.2,
    transform=map_projection,
    zorder=4,
)
map_ax.text(
    bordeaux_lon + 0.02,
    bordeaux_lat + 0.02,
    "Bordeaux",
    fontsize=18,
    color="black",
    ha="left",
    va="bottom",
    bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.7, "pad": 2},
    transform=map_projection,
    zorder=5,
)

map_ax.set_title(f"{target_day.date()}", fontsize=16)
map_ax.set_xlabel("Longitude", fontsize=16)
map_ax.set_ylabel("Latitude", fontsize=16)
gridlines = map_ax.gridlines(
    draw_labels=True,
    linewidth=0.4,
    alpha=0.5,
    linestyle="--",
)
gridlines.top_labels = False
gridlines.right_labels = False
gridlines.xlabel_style = {"size": 16}
gridlines.ylabel_style = {"size": 16}

# Colorbar axes in map-axes coordinates, so it matches the map height exactly
colorbar_ax = map_ax.inset_axes([1.03, 0, 0.04, 1])
colorbar = figure.colorbar(raster, cax=colorbar_ax, orientation="vertical")
colorbar.set_label("Risico op luchtvervuiling", fontsize=22)
colorbar.ax.tick_params(labelsize=22)

output_svg = figure_folder / f"AAI_{target_day:%Y%m%d}.svg"
figure.savefig(output_svg, format="svg", bbox_inches="tight")
plt.show()
plt.close(figure)
print(f"Saved {output_svg}")

In [ ]:
from IPython.display import display

# Child-friendly version of the figure above: no numbers on the colorbar,
# just "Laag" at the bottom, "Hoog" at the top and an arrow from low to high.
low_label, high_label = "Laag", "Hoog"

# Remove earlier additions so re-running this cell does not stack them
for artist in [text for text in colorbar.ax.texts if text.get_gid() == "kids_colorbar"]:
    artist.remove()

colorbar.set_ticks([])
colorbar.ax.text(0.5, -0.01, low_label, transform=colorbar.ax.transAxes,
                 ha="center", va="top", fontsize=22, gid="kids_colorbar")
colorbar.ax.text(0.5, 1.01, high_label, transform=colorbar.ax.transAxes,
                 ha="center", va="bottom", fontsize=22, gid="kids_colorbar")
colorbar.ax.annotate(
    "",
    xy=(2.0, 0.97),
    xytext=(2.0, 0.03),
    xycoords="axes fraction",
    arrowprops={"arrowstyle": "-|>", "linewidth": 3, "color": "black", "mutation_scale": 30},
    annotation_clip=False,
    gid="kids_colorbar",
)
colorbar.ax.yaxis.labelpad = 45  # keep the colorbar title clear of the arrow

kids_svg = output_svg.with_name(f"{output_svg.stem}_kids.svg")
figure.savefig(kids_svg, format="svg", bbox_inches="tight")
display(figure)
print(f"Saved {kids_svg}")